In [2]:
import pandas as pd
%matplotlib inline
import tangos
import tangos.examples.mergers as mergers
import pynbody

import numpy as np
import h5py
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
h = 0.6776942783267969

from tangos_halo_module.halo_properties import track_halo_property, get_timesteps, ID_to_sim_halo_snap, infall_final_n_particles, infall_final_coordinates, apocentric_distance, disruption_time, accretion_time, orbit_interpolation, infall_velocity, quenching_time, max_sSFR_time, max_mass_time 
from tangos_halo_module.path import get_file_path, get_halo_snap_num, read_file
from tangos_halo_module.halos import ID_to_tangos_halo, get_survivors, get_main_progenitor_branch, get_zombies, get_host, get_survivor_IDs, get_zombie_IDs, blockPrint, enablePrint, tangos_to_pynbody_halo

In [3]:
ids = np.loadtxt('ids100.txt')
ids=[int(i) for i in ids]
ids

[14840961025,
 14832642,
 14840964127,
 32913443,
 14840961061,
 14812709,
 329022513,
 1484096567,
 3294096956,
 32940964414,
 329022530,
 329409616,
 329038417,
 329409620,
 329038430,
 329127010,
 329038438,
 32904809,
 329409643,
 148409620589,
 329022579,
 14840962,
 14840963,
 14840961156,
 14840964,
 14840965,
 14840966,
 14840967,
 14812804,
 1484096138,
 329409673,
 329409675,
 32906374,
 32940965015,
 14840961179,
 14840961693,
 329409696,
 3294096555,
 1484096179,
 1484096194,
 14829762,
 3294096585,
 329067210,
 1484096204,
 1484096210,
 32933603,
 3290192110,
 3290192111,
 148409613555,
 148409619718,
 148153610,
 148409610,
 148409611,
 148409612,
 148128013,
 1484096271,
 14840965903,
 148409617,
 148110610,
 148115218,
 148409621,
 1484096278,
 148409622,
 148409632,
 148409633,
 148409634,
 148409635,
 1484096803,
 32924962,
 148409639,
 148409641,
 148409651,
 148409653,
 148409655,
 14840964409,
 148409663,
 148409668,
 1484096266,
 1484096331,
 148409676,
 148409677

In [3]:
idx=[str(int(i)) for i in ids]
idx.sort()
h148_ids = idx[:77]
h329_ids = idx[77:]
print(h148_ids)
print(h329_ids)

['148110610', '14811066', '148115218', '14812709', '148128013', '14812804', '14814089', '148153610', '14829762', '14832642', '148409610', '14840961025', '14840961061', '148409611', '14840961156', '14840961179', '148409612', '148409612211', '148409613555', '14840961379', '1484096138', '14840961413', '14840961467', '14840961693', '148409617', '1484096179', '14840961883', '14840961900', '1484096194', '148409619718', '14840962', '14840962026', '1484096204', '148409620589', '148409621', '1484096210', '148409622', '14840962413', '1484096266', '1484096271', '1484096278', '14840962927', '14840963', '148409632', '148409633', '1484096331', '148409634', '148409635', '1484096374', '1484096387', '148409639', '14840964', '14840964022', '148409641', '1484096412', '14840964127', '1484096422', '14840964409', '1484096459', '1484096478', '14840964944', '14840965', '148409651', '148409653', '148409655', '1484096567', '14840965903', '14840966', '148409663', '148409668', '14840967', '14840967007', '14840967

### Get MPB ids & Store them

In [5]:
# for sim in ['h329', 'h148']:
#     if sim=='h329':
#         idx = h329_ids
#     else:
#         idx = h148_ids
#     column_names = ['snapshot']+idx
#     df = pd.DataFrame(columns = column_names)
#     df.to_csv(str(sim)+'MPB_ids.csv', index=False)
#     ts = get_timesteps(simulation=sim, resolution=100)[3]
#     df['snapshot'] = ts
#     df.to_csv(str(sim)+'MPB_ids.csv', index=False)
#     print('initialized ', sim)

resolution=100
def store_MPB_ids(ID=0):
    if ID != 0:
        print('Started ', ID)
        simulation, status, halo_id, snap_num = ID_to_sim_halo_snap(ID=ID)
        print(simulation, status, halo_id, snap_num)
        tangos_halo = ID_to_tangos_halo(ID=ID, resolution=resolution)
    
    # Satellite MPB ids
    MPB, MPB_ids = get_main_progenitor_branch(tangos_halo=tangos_halo, simulation=simulation, resolution=resolution)
    print(MPB_ids)
    print(MPB)
    # Store and recall
    # Host MPB and ids
    host = get_host(simulation=simulation, resolution=resolution)
    host_MPB, host_ids = get_main_progenitor_branch(tangos_halo=host, simulation=simulation, 
                                                    resolution=resolution)
    print(host_ids)
    # Only consider halo while separate from host
    for i, idx in enumerate(MPB_ids):
        if idx in host_ids:
            MPB_ids[i]=0
            
    MPB_ids[MPB_ids==host_ids] = 0
    print('Separate from host', MPB_ids)

    for i in range(len(MPB_ids)):
        if MPB_ids[i]!=0:
            if len(str(MPB[i]))>6:
                MPB_ids[i]=get_halo_snap_num(tangos_halo=MPB[i])[2]
            else: 
                MPB_ids[i]=0
#     dat = {str(ID): MPB_ids,}
#     df = pd.DataFrame(data=dat)
    df = pd.read_csv(str(simulation)+'MPB_ids.csv')
    df[str(ID)] = MPB_ids
    df.to_csv(str(simulation)+'MPB_ids.csv', index=False)
    print(df)
    print('Finished ', ID)
    
completed_ids = np.zeros(0)

In [11]:
# count=0
# done=0
# while done<124:
#     try:
#         print('Watch me go for the ', count, 'th time...')
#         for idx in ids:
#             if idx in completed_ids:
#                 print('Already Exists ', idx)
#                 done=len(completed_ids)
#                 print('Completed Count --> ', done)
#                 pass
#             else:
#                 print('Started ', idx)
#                 store_MPB_ids(ID=idx)
#                 completed_ids = np.concatenate((completed_ids, [idx]), axis=None)
#                 print('Finished ', idx)
#                 done=len(completed_ids)
#                 print('Completed Count --> ', done)
#     except: 
#         count+=1
#         continue

Watch me go for the  0 th time...
Already Exists  14840961025
Completed Count -->  77
Already Exists  14832642
Completed Count -->  77
Already Exists  14840964127
Completed Count -->  77
Already Exists  32913443
Completed Count -->  77
Already Exists  14840961061
Completed Count -->  77
Already Exists  14812709
Completed Count -->  77
Already Exists  329022513
Completed Count -->  77
Already Exists  1484096567
Completed Count -->  77
Already Exists  3294096956
Completed Count -->  77
Already Exists  32940964414
Completed Count -->  77
Already Exists  329022530
Completed Count -->  77
Already Exists  329409616
Completed Count -->  77
Already Exists  329038417
Completed Count -->  77
Already Exists  329409620
Completed Count -->  77
Already Exists  329038430
Completed Count -->  77
Already Exists  329127010
Completed Count -->  77
Already Exists  329038438
Completed Count -->  77
Already Exists  32904809
Completed Count -->  77
Already Exists  329409643
Completed Count -->  77
Already Ex

Watch me go for the  1 th time...
Already Exists  14840961025
Completed Count -->  78
Already Exists  14832642
Completed Count -->  78
Already Exists  14840964127
Completed Count -->  78
Already Exists  32913443
Completed Count -->  78
Already Exists  14840961061
Completed Count -->  78
Already Exists  14812709
Completed Count -->  78
Already Exists  329022513
Completed Count -->  78
Already Exists  1484096567
Completed Count -->  78
Already Exists  3294096956
Completed Count -->  78
Already Exists  32940964414
Completed Count -->  78
Already Exists  329022530
Completed Count -->  78
Already Exists  329409616
Completed Count -->  78
Already Exists  329038417
Completed Count -->  78
Already Exists  329409620
Completed Count -->  78
Already Exists  329038430
Completed Count -->  78
Already Exists  329127010
Completed Count -->  78
Already Exists  329038438
Completed Count -->  78
Already Exists  32904809
Completed Count -->  78
Already Exists  329409643
Completed Count -->  78
Already Ex

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, '1481024107', '1481106108', '1481152105', '1481270108', '1481280108', '148140894', '148147696', '148153693', '148166492', '148174096', '1481920117', '1482048113', '1482088120', '1482112120', '1482208126', '1482304130', '1482400130', '1482496131', '1482555130', '1482592130', '1482688128', '1482784124', '1482880125', '1482976122', '1483072118', '1483168119', '1483195119', '1483264118', '1483360293', '1483456276', '1483552282', '1483606289', '1483648293', '1483671292', '1483744299', '1483760304', '1483810309', '1483840315', '1483936328', '1484032331', '1484096331']
    snapshot  148110610  14811066  148115218  14812709  148128013  14812804  \
0         71          0       NaN          0         0          0         0   
1        108          0       NaN          0         0          0         0   
2        128          0       NaN          0         0          0         0   
3        139          0       NaN          0         0         

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, '148102478', '148110676', '148115274', '148127070', '148128069', '1481408101', '148147690', '148153683', '148166489', '148174092', '148192093', '148204891', '148208892', '148211292', '148220894', '148230496', '148240096', '148249696', '148255597', '148259296', '148268897', '148278496', '148288095', '148297694', '148307291', '148316888', '148319588', '148326484', '148336080', '148345680', '148355277', '148360680', '148364880', '148367180', '148374477', '148376077', '148381077', '148384077', '148393676', '148403276', '148409676']
[-1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0
 -1.0 -1.0
 <Halo 'snapshots/h148.cosmo50PLK.6144g3HbwK1BH.001024/halo_78' | NDM=76880 Nstar=510 Ngas=2266>
 <Halo 'snapshots/h148.cosmo50PLK.6144g3HbwK1BH.001106/halo_76' | NDM=77604 Nstar=510 Ngas=1139>
 <Halo 'snapshots/h148.cosmo50PLK.6144g3HbwK1BH.001152/halo_74' | NDM=76982 Nstar=510 Ngas=1194>
 <Halo 'snapshots/h148.cosmo50PLK.61

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, '148102478', '148110676', '148115274', '148127070', '148128069', '1481408101', '148147690', '148153683', '148166489', '148174092', '148192093', '148204891', '148208892', '148211292', '148220894', '148230496', '148240096', '148249696', '148255597', '148259296', '148268897', '148278496', '148288095', '148297694', '148307291', '148316888', '148319588', '148326484', '148336080', '148345680', '148355277', '148360680', '148364880', '148367180', '148374477', '148376077', '148381077', '148384077', '148393676', '148403276', '148409676']
[-1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0
 -1.0 -1.0
 <Halo 'snapshots/h148.cosmo50PLK.6144g3HbwK1BH.001024/halo_78' | NDM=76880 Nstar=510 Ngas=2266>
 <Halo 'snapshots/h148.cosmo50PLK.6144g3HbwK1BH.001106/halo_76' | NDM=77604 Nstar=510 Ngas=1139>
 <Halo 'snapshots/h148.cosmo50PLK.6144g3HbwK1BH.001152/halo_74' | NDM=76982 Nstar=510 Ngas=1194>
 <Halo 'snapshots/h148.cosmo50PLK.61

...

['329007153', '3290096158', '3290108155', '3290139385', '3290188324', '3290192316', '3290225281', '3290275295', '3290288301', '3290347322', '3290384328', '3290457387', '3290480429', '3290576585', '3290637662', '3290672696', '3290768792', '3290777801', '3290864861', '3290960921', '3290974925', '32910561163', '32911061242', '32911521233', '32912481380', '32912701390', '32913441393', '32914401366', '32914761361', '32915361343', '32916321341', '32917281324', '32917401323', '32918241311', '32919201299', '32920161274', '32920881270', '32921121265', '32922081248', '32923041245', '32924001847', '32924961965', '32925551966', '32925921993', '32926881981', '32927842006', '32928801984', '32929761961', '32930721935', '32931682082', '32931952970', '32932643278', '32933603297', '32934563330', '32935523357', '32936063340', '32936483368', '32937443379', '32938403362', '32939363311', '32940323293', '32940963299']
[ <Halo 'snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000071/halo_53' | NDM=531 Nstar=0 Ngas=432

[0, 0, 0, '32901394122', '329018834', '329019234', '329022535', '329027518', '329028819', '329034719', '329038418', '329045715', '329048015', '329057612', '329063710', '32906729', '32907681', '32907771', '32908641', '32909601', '32909741', '32910561', '32911061', '32911521', '32912481', '32912701', '32913441', '32914401', '32914761', '32915361', '32916321', '32917281', '32917401', '32918241', '32919201', '32920161', '32920881', '32921121', '32922081', '32923041', '32924001', '32924961', '32925551', '32925921', '32926881', '32927841', '32928801', '32929761', '32930721', '32931681', '32931951', '32932641', '32933601', '32934561', '32935521', '32936061', '32936481', '32937441', '32938401', '32939361', '32940321', '32940961']
[-1.0 -1.0 -1.0
 <Halo 'snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000139/halo_4122' | NDM=101 Nstar=0 Ngas=0>
 <Halo 'snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000188/halo_34' | NDM=20124 Nstar=1685 Ngas=9221>
 <Halo 'snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000192/halo_34

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, '148102462', '148110668', '148115265', '148127096', '1481280102', '1481408130', '1481476145', '1481536155', '1481664159', '1481740173', '1481920230', '1482048273', '1482088283', '1482112288', '1482208307', '1482304341', '1482400419', '1482496523', '1482555576', '1482592614', '1482688671', '1482784721', '1482880771', '1482976805', '1483072820', '1483168861', '1483195882', '1483264896', '1483360938', '1483456967', '1483552989', '1483606997', '14836481000', '14836711000', '14837441023', '14837601028', '14838101043', '14838401046', '14839361194', '14840321568', '14840962026']
[-1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0
 -1.0 -1.0
 <Halo 'snapshots/h148.cosmo50PLK.6144g3HbwK1BH.001024/halo_62' | NDM=85698 Nstar=1373 Ngas=17880>
 <Halo 'snapshots/h148.cosmo50PLK.6144g3HbwK1BH.001106/halo_68' | NDM=80996 Nstar=1374 Ngas=16420>
 <Halo 'snapshots/h148.cosmo50PLK.6144g3HbwK1BH.001152/halo_65' | NDM=77724 Nstar=13

['329007113', '329009617', '329010817', '3290139109', '329018881', '329019272', '329022549', '329027538', '329028840', '329034745', '32903841', '32904571', '32904801', '32905761', '32906371', '32906721', '32907681', '32907771', '32908641', '32909601', '32909741', '32910561', '32911061', '32911521', '32912481', '32912701', '32913441', '32914401', '32914761', '32915361', '32916321', '32917281', '32917401', '32918241', '32919201', '32920161', '32920881', '32921121', '32922081', '32923041', '32924001', '32924961', '32925551', '32925921', '32926881', '32927841', '32928801', '32929761', '32930721', '32931681', '32931951', '32932641', '32933601', '32934561', '32935521', '32936061', '32936481', '32937441', '32938401', '32939361', '32940321', '32940961']
[ <Halo 'snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000071/halo_13' | NDM=846 Nstar=0 Ngas=672>
 <Halo 'snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000096/halo_17' | NDM=2069 Nstar=0 Ngas=1638>
 <Halo 'snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000108/ha

[0, 0, 0, '329013943', '329018815', '329019210', '32902255', '32902754', '32902884', '32903474', '32903844', '32904576', '32904807', '32905766', '32906376', '32906724', '32907684', '32907774', '32908641', '32909601', '32909741', '32910561', '32911061', '32911521', '32912481', '32912701', '32913441', '32914401', '32914761', '32915361', '32916321', '32917281', '32917401', '32918241', '32919201', '32920161', '32920881', '32921121', '32922081', '32923041', '32924001', '32924961', '32925551', '32925921', '32926881', '32927841', '32928801', '32929761', '32930721', '32931681', '32931951', '32932641', '32933601', '32934561', '32935521', '32936061', '32936481', '32937441', '32938401', '32939361', '32940321', '32940961']
[-1.0 -1.0 -1.0
 <Halo 'snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000139/halo_43' | NDM=7077 Nstar=202 Ngas=4119>
 <Halo 'snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000188/halo_15' | NDM=39515 Nstar=432 Ngas=24358>
 <Halo 'snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000192/halo_10' | NDM

In [4]:
# simulation='h148'
# resolution=100
# host = get_host(simulation=simulation, resolution=resolution)
# host_MPB, host_ids = get_main_progenitor_branch(tangos_halo=host, simulation=simulation, 
#                                                 resolution=resolution)
# host_ids=[int(i) for i in host_ids]
# df=df.replace(host_ids, 0)
# df

pd.set_option("display.max_rows", None, "display.max_columns", None)
df = pd.read_csv('h148MPB_ids.csv')
# df.to_csv('h148MPB_ids.csv', index=False)
df

,snapshot,148110610,14811066,148115218,14812709,148128013,14812804,14814089,148153610,14829762,14832642,148409610,14840961025,14840961061,148409611,14840961156,14840961179,148409612,148409612211,148409613555,14840961379,1484096138,14840961413,14840961467,14840961693,148409617,1484096179,14840961883,14840961900,1484096194,148409619718,14840962,14840962026,1484096204,148409620589,148409621,1484096210,148409622,14840962413,1484096266,1484096271,1484096278,14840962927,14840963,148409632,148409633,1484096331,148409634,148409635,1484096374,1484096387,148409639,14840964,14840964022,148409641,1484096412,14840964127,1484096422,14840964409,1484096459,1484096478,14840964944,14840965,148409651,148409653,148409655,1484096567,14840965903,14840966,148409663,148409668,14840967,14840967007,148409676,148409677,1484096803,14840968565,1484096857
0,71,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,108,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,128,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,139,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,188,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,225,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,256,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,275,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,347,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,384,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### Get & Store sSFR

In [5]:
# for sim in ['h329', 'h148']:
#     if sim=='h329':
#         idx = h329_ids
#     else:
#         idx = h148_ids
#     for what in ['SFR', 'sSFR', 'mass', 'new_mass']:
#         column_names = ['snapshot']+idx
#         df = pd.DataFrame(columns = column_names)
#         df.to_csv(str(sim)+'MPB_'+ str(what) +'.csv', index=False)
#         ts = get_timesteps(simulation=sim, resolution=100)[3]
#         df['snapshot'] = ts
#         df.to_csv(str(sim)+'MPB_'+ str(what) +'.csv', index=False)
#         print(what, ' initialized ', sim)

SFR  initialized  h329
sSFR  initialized  h329
mass  initialized  h329
new_mass  initialized  h329
SFR  initialized  h148
sSFR  initialized  h148
mass  initialized  h148
new_mass  initialized  h148


In [ ]:
for sim in ['h329', 'h148']:
    print('started simulation ', sim)
    
    df = pd.read_csv(str(sim)+'MPB_ids.csv')
    IDs = np.asarray(df.columns)[1:]
    print('my IDs are ', IDs)
    ts = get_timesteps(simulation=sim, resolution=100)[3]
    print('the timesteps are ', ts)
    
    
    
    #Read in new dfs
    df_SFR = pd.read_csv(str(sim)+'MPB_SFR.csv')
    df_sSFR = pd.read_csv(str(sim)+'MPB_sSFR.csv')
    df_mass = pd.read_csv(str(sim)+'MPB_mass.csv')
    df_new_mass = pd.read_csv(str(sim)+'MPB_new_mass.csv')
    
    #Iterate through each timestep
    for index in range(0, len(ts)):
        #index is #row
        snapnum = ts[index]
        print('starting with timestep ', snapnum)
        
        halo_ids = np.asarray(df.loc[index][1:])
        print('for timestep ', snapnum, ' my halo IDs are ', halo_ids)
    
        if sim=='h329':
            snapshot = pynbody.load('/home/dp1144/tangos_simulations/'+ str(sim) +'/snapshots/'+ str(sim) +'.cosmo50PLK.6144g5HbwK1BH.00'+ str(snapnum))
        else:
            snapshot = pynbody.load('/home/dp1144/tangos_simulations/'+ str(sim) +'/snapshots/'+ str(sim) +'.cosmo50PLK.6144g3HbwK1BH.00'+ str(snapnum))
        snapshot.physical_units()
        all_halos = snapshot.halos()
        print(snapshot)
        print(all_halos)
        
        count=0 #This is #column-1
        
        #Iterate through each halo
        for idx in halo_ids:
            print('started [', index, ', ', count, ']')
            ID = IDs[count]
            print('my ID is ', ID)
            print('')
            if idx==0:
                #set everything to -1 and move on
                df_SFR.iat[index, count+1] = -1
                df_sSFR.iat[index, count+1] = -1
                df_mass.iat[index, count+1] = -1
                df_new_mass.iat[index, count+1] = -1
            else:
                idx=int(idx)
                simulation, status, halo_id, snap_num = ID_to_sim_halo_snap(ID=idx)
                halo=all_halos[int(halo_id)]
                print(halo)
                #Do the calculation
                mass = np.asarray(halo.s['mass'].in_units('Msol'))
                age = halo.properties['time'].in_units('Gyr') - halo.s['tform'].in_units('Gyr') #SimArray in Gyr
                new_mass = np.sum(mass[age <= 0.1]) # Formed within last 100 Myr
                total_mass = np.sum(mass)
                SFR = new_mass/1e8
                if SFR==0:
                    sSFR=0
                else:
                    sSFR = SFR/total_mass
                
                print('SFR: ', SFR)
                print('sSFR: ', sSFR)
                print('mass: ', total_mass)
                print('new mass: ', new_mass)
                
                #add data in dfs
                df_SFR.iat[index, count+1] = SFR
                df_sSFR.iat[index, count+1] = sSFR
                df_mass.iat[index, count+1] = total_mass
                df_new_mass.iat[index, count+1] = new_mass
            print('finished [', index, ', ', count+1, ']')     

            df_SFR.to_csv(str(sim)+'MPB_SFR.csv', index=False)
            df_sSFR.to_csv(str(sim)+'MPB_sSFR.csv', index=False)
            df_mass.to_csv(str(sim)+'MPB_mass.csv', index=False)
            df_new_mass.to_csv(str(sim)+'MPB_new_mass.csv', index=False)
            
            print('saved [', index, ', ', count, ']')
            
            #Move on to next id == next column
            count+=1
        
        print('finished timestep ', snapnum)
    print('finished simulation ', sim)
print('FINALLY WE ARE DONE!!!')

    

started simulation  h329
my IDs are  ['3290188153' '3290192110' '3290192111' '329019225' '329019230' '329022513'
 '329022530' '329022579' '329034745' '329038417' '329038430' '329038438'
 '329048011' '329048022' '329048034' '32904809' '32906374' '329067210'
 '32906729' '32907774' '32908648' '329096015' '32909747' '329127010'
 '32913443' '32914768' '32924962' '32930722' '32933603' '329409616'
 '32940961630' '32940961648' '329409620' '32940963299' '3294096335'
 '3294096350' '329409643' '32940964414' '32940965015' '3294096555'
 '3294096585' '329409673' '329409675' '3294096768' '32940969' '3294096956'
 '329409696']
the timesteps are  ['0071', '0096', '0108', '0139', '0188', '0192', '0225', '0275', '0288', '0347', '0384', '0457', '0480', '0576', '0637', '0672', '0768', '0777', '0864', '0960', '0974', '1056', '1106', '1152', '1248', '1270', '1344', '1440', '1476', '1536', '1632', '1728', '1740', '1824', '1920', '2016', '2088', '2112', '2208', '2304', '2400', '2496', '2555', '2592', '2688', '2

SFR:  0.0
sSFR:  0
mass:  0.0
new mass:  0.0
finished [ 1 ,  13 ]
saved [ 1 ,  12 ]
started [ 1 ,  13 ]
my ID is  329048022

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000096:halo_3" len=32927>
SFR:  0.00835871609749
sSFR:  8.34634876523e-09
mass:  1001481.76557
new mass:  835871.609749
finished [ 1 ,  14 ]
saved [ 1 ,  13 ]
started [ 1 ,  14 ]
my ID is  329048034

finished [ 1 ,  15 ]
saved [ 1 ,  14 ]
started [ 1 ,  15 ]
my ID is  32904809

finished [ 1 ,  16 ]
saved [ 1 ,  15 ]
started [ 1 ,  16 ]
my ID is  32906374

finished [ 1 ,  17 ]
saved [ 1 ,  16 ]
started [ 1 ,  17 ]
my ID is  329067210

finished [ 1 ,  18 ]
saved [ 1 ,  17 ]
started [ 1 ,  18 ]
my ID is  32906729

finished [ 1 ,  19 ]
saved [ 1 ,  18 ]
started [ 1 ,  19 ]
my ID is  32907774

finished [ 1 ,  20 ]
saved [ 1 ,  19 ]
started [ 1 ,  20 ]
my ID is  32908648

finished [ 1 ,  21 ]
saved [ 1 ,  20 ]
started [ 1 ,  21 ]
my ID is  329096015

finished [ 1 ,  22 ]
saved [ 1 , 

SFR:  0.0
sSFR:  0
mass:  0.0
new mass:  0.0
finished [ 2 ,  13 ]
saved [ 2 ,  12 ]
started [ 2 ,  13 ]
my ID is  329048022

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000108:halo_2" len=38202>
SFR:  0.0016599709931
sSFR:  1.45330598842e-09
mass:  1142203.36689
new mass:  165997.09931
finished [ 2 ,  14 ]
saved [ 2 ,  13 ]
started [ 2 ,  14 ]
my ID is  329048034

finished [ 2 ,  15 ]
saved [ 2 ,  14 ]
started [ 2 ,  15 ]
my ID is  32904809

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000108:halo_1510" len=138>
SFR:  0.0
sSFR:  0
mass:  0.0
new mass:  0.0
finished [ 2 ,  16 ]
saved [ 2 ,  15 ]
started [ 2 ,  16 ]
my ID is  32906374

finished [ 2 ,  17 ]
saved [ 2 ,  16 ]
started [ 2 ,  17 ]
my ID is  329067210

finished [ 2 ,  18 ]
saved [ 2 ,  17 ]
started [ 2 ,  18 ]
my ID is  32906729

finished [ 2 ,  19 ]
saved [ 2 ,  18 ]
started [ 2 ,  19 ]
my ID is  32907774

finished [ 2 ,  20 ]
saved [ 2 ,  1

SFR:  0.00067489423274
sSFR:  1e-08
mass:  67489.423274
new mass:  67489.423274
finished [ 3 ,  9 ]
saved [ 3 ,  8 ]
started [ 3 ,  9 ]
my ID is  329038417

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000139:halo_264" len=1343>
SFR:  0.0
sSFR:  0
mass:  0.0
new mass:  0.0
finished [ 3 ,  10 ]
saved [ 3 ,  9 ]
started [ 3 ,  10 ]
my ID is  329038430

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000139:halo_931" len=458>
SFR:  0.0
sSFR:  0
mass:  0.0
new mass:  0.0
finished [ 3 ,  11 ]
saved [ 3 ,  10 ]
started [ 3 ,  11 ]
my ID is  329038438

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000139:halo_49" len=10314>
SFR:  0.00106684478338
sSFR:  1e-08
mass:  106684.478338
new mass:  106684.478338
finished [ 3 ,  12 ]
saved [ 3 ,  11 ]
started [ 3 ,  12 ]
my ID is  329048011

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.0001

SFR:  1.9560095703e-05
sSFR:  1e-08
mass:  1956.0095703
new mass:  1956.0095703
finished [ 3 ,  45 ]
saved [ 3 ,  44 ]
started [ 3 ,  45 ]
my ID is  3294096956

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000139:halo_40" len=12713>
SFR:  0.00020984292388
sSFR:  1e-08
mass:  20984.292388
new mass:  20984.292388
finished [ 3 ,  46 ]
saved [ 3 ,  45 ]
started [ 3 ,  46 ]
my ID is  329409696

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000139:halo_60" len=8789>
SFR:  0.00145184087804
sSFR:  1e-08
mass:  145184.087804
new mass:  145184.087804
finished [ 3 ,  47 ]
saved [ 3 ,  46 ]
finished timestep  0139
starting with timestep  0188
for timestep  0188  my halo IDs are  [ 3290188153  3290188114  3290188111   329018827   329018833   329018813
   329018845   329018866   329018881   329018840  3290188477   329018830
    32901885   329018810   329018844 32901882712    32901881   329018811
   329018834   3290188

...

SFR:  8.10489523096e-05
sSFR:  1e-08
mass:  8104.89523096
new mass:  8104.89523096
finished [ 5 ,  10 ]
saved [ 5 ,  9 ]
started [ 5 ,  10 ]
my ID is  329038430

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000192:halo_418" len=1357>
SFR:  0.0
sSFR:  0
mass:  0.0
new mass:  0.0
finished [ 5 ,  11 ]
saved [ 5 ,  10 ]
started [ 5 ,  11 ]
my ID is  329038438

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000192:halo_28" len=39758>
SFR:  0.000748164238452
sSFR:  4.45969897509e-09
mass:  167761.152183
new mass:  74816.4238452
finished [ 5 ,  12 ]
saved [ 5 ,  11 ]
started [ 5 ,  12 ]
my ID is  329048011

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000192:halo_4" len=153213>
SFR:  0.0450382474685
sSFR:  4.45042746396e-09
mass:  10119982.3687
new mass:  4503824.74685
finished [ 5 ,  13 ]
saved [ 5 ,  12 ]
started [ 5 ,  13 ]
my ID is  329048022

<SimSnap "/home/dp1144/

SFR:  0.0
sSFR:  0
mass:  35757.5142991
new mass:  0.0
finished [ 5 ,  44 ]
saved [ 5 ,  43 ]
started [ 5 ,  44 ]
my ID is  32940969

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000192:halo_20" len=46748>
SFR:  0.000545537278673
sSFR:  1.83360658744e-09
mass:  297521.443483
new mass:  54553.7278673
finished [ 5 ,  45 ]
saved [ 5 ,  44 ]
started [ 5 ,  45 ]
my ID is  3294096956

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000192:halo_61" len=16259>
SFR:  9.31761051105e-05
sSFR:  1.1766676319e-09
mass:  79186.426638
new mass:  9317.61051105
finished [ 5 ,  46 ]
saved [ 5 ,  45 ]
started [ 5 ,  46 ]
my ID is  329409696

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000192:halo_81" len=10848>
SFR:  0.0
sSFR:  0
mass:  132155.132147
new mass:  0.0
finished [ 5 ,  47 ]
saved [ 5 ,  46 ]
finished timestep  0192
starting with timestep  0225
for timestep  0225  my halo 

SFR:  0.0108644934947
sSFR:  2.99000507115e-09
mass:  3633603.70173
new mass:  1086449.34947
finished [ 6 ,  29 ]
saved [ 6 ,  28 ]
started [ 6 ,  29 ]
my ID is  329409616

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000225:halo_78" len=13351>
SFR:  8.82966793576e-06
sSFR:  4.76608762532e-10
mass:  18526.0293765
new mass:  882.966793576
finished [ 6 ,  30 ]
saved [ 6 ,  29 ]
started [ 6 ,  30 ]
my ID is  32940961630

finished [ 6 ,  31 ]
saved [ 6 ,  30 ]
started [ 6 ,  31 ]
my ID is  32940961648

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000225:halo_644" len=1095>
SFR:  0.0
sSFR:  0
mass:  11127.1589187
new mass:  0.0
finished [ 6 ,  32 ]
saved [ 6 ,  31 ]
started [ 6 ,  32 ]
my ID is  329409620

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000225:halo_60" len=20426>
SFR:  0.000256080430434
sSFR:  4.21836913477e-09
mass:  60706.0269627
new mass:  25608.0430

SFR:  0.0477886759481
sSFR:  1.29518859938e-09
mass:  36897078.905
new mass:  4778867.59481
finished [ 7 ,  17 ]
saved [ 7 ,  16 ]
started [ 7 ,  17 ]
my ID is  329067210

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000275:halo_10" len=188385>
SFR:  0.00948186234217
sSFR:  2.48389449478e-09
mass:  3817336.99322
new mass:  948186.234217
finished [ 7 ,  18 ]
saved [ 7 ,  17 ]
started [ 7 ,  18 ]
my ID is  32906729

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000275:halo_18" len=91990>
SFR:  0.00233096723325
sSFR:  1.54562819766e-09
mass:  1508103.46031
new mass:  233096.723325
finished [ 7 ,  19 ]
saved [ 7 ,  18 ]
started [ 7 ,  19 ]
my ID is  32907774

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000275:halo_4" len=449656>
SFR:  0.0371103266652
sSFR:  2.60883240377e-09
mass:  14224879.5329
new mass:  3711032.66652
finished [ 7 ,  20 ]
saved [ 7 ,  19 ]
started

<SimSnap "/home/dp1144/tangos_simulations/h329/snapshots/h329.cosmo50PLK.6144g5HbwK1BH.000288:halo_40" len=54077>


In [4]:
pd.set_option("display.max_rows", None, "display.max_columns", None)

df = pd.read_csv('h329MPB_SFR.csv')
# df.to_csv('h148MPB_ids.csv', index=False)
df

,snapshot,3290188153,3290192110,3290192111,329019225,329019230,329022513,329022530,329022579,329034745,329038417,329038430,329038438,329048011,329048022,329048034,32904809,32906374,329067210,32906729,32907774,32908648,329096015,32909747,329127010,32913443,32914768,32924962,32930722,32933603,329409616,32940961630,32940961648,329409620,32940963299,3294096335,3294096350,329409643,32940964414,32940965015,3294096555,3294096585,329409673,329409675,3294096768,32940969,3294096956,329409696
0,71,-1.000000,-1.0,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.0,-1.000000,-1.000000,-1.0,-1.0,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.000000,-1.000000,-1.0,-1.0,-1.000000,-1.000000,-1.000000
1,96,0.000000,0.0,0.000000,-1.000000,-1.000000,0.000000,0.000000,-1.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,0.008359,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,-1.000000,-1.000000,-1.0,-1.000000,-1.000000,-1.0,0.0,0.000000,0.000081,-1.000000,0.000008,-1.000000,-1.0,0.0,-1.0,0.000000,0.000008,-1.0,0.0,-1.000000,-1.000000,0.000000
2,108,0.000000,0.0,0.000000,-1.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,0.001660,-1.000000,0.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,-1.000000,-1.000000,-1.0,0.000000,-1.000000,-1.0,0.0,0.000000,0.000000,-1.000000,0.000000,-1.000000,-1.0,0.0,-1.0,0.000000,0.000000,-1.0,0.0,-1.000000,-1.000000,0.000000
3,139,0.000073,0.0,0.001565,0.000010,0.000000,0.005129,0.004052,0.000000,0.000675,0.000000,0.000000,0.001067,0.008408,0.007506,0.000000,0.000000,0.097457,0.001760,0.000000,0.001672,0.000169,0.000169,0.000000,0.000000,0.035935,0.000000,0.000059,-1.0,0.000256,-1.000000,-1.0,0.0,0.000010,0.000000,0.000000,0.000000,0.002397,-1.0,0.0,-1.0,0.000000,0.000000,-1.0,0.0,0.000020,0.000210,0.001452
4,188,0.000008,0.0,0.000009,0.002316,0.000587,0.000000,0.000000,0.000697,0.000000,0.000055,0.000000,0.000382,0.048483,0.003783,0.000418,0.000000,0.046398,0.014665,0.001444,0.000486,0.001358,0.001358,0.000180,0.000180,0.022865,0.000000,0.002140,-1.0,0.023669,0.000162,-1.0,0.0,0.000266,0.000000,0.000009,0.000000,0.001021,-1.0,0.0,-1.0,0.000068,0.000000,-1.0,0.0,0.000514,0.000094,0.000000
5,192,-1.000000,0.0,0.000009,0.002230,0.000455,0.000000,0.000000,0.000622,0.000000,0.000081,0.000000,0.000748,0.045038,0.002855,0.000476,0.000000,0.049565,0.014527,0.001418,0.001560,0.001418,0.001418,0.000148,0.000148,0.022788,0.000000,0.006620,-1.0,0.021895,0.000160,-1.0,0.0,0.000238,0.000000,0.000017,0.000000,0.001142,-1.0,0.0,-1.0,0.000059,0.000000,-1.0,0.0,0.000546,0.000093,0.000000
6,225,-1.000000,-1.0,-1.000000,-1.000000,-1.000000,0.000008,0.000089,0.000049,0.000000,0.002894,0.000000,0.002767,0.012994,0.012453,0.001137,0.000000,0.045980,0.009490,0.000644,0.049717,0.001937,0.001937,0.000201,0.000201,0.036998,0.000000,0.011853,-1.0,0.010864,0.000009,-1.0,0.0,0.000256,0.000000,0.000103,0.000000,0.004887,-1.0,0.0,-1.0,0.000033,0.000000,0.0,0.0,0.001436,0.000232,0.000000
7,275,-1.000000,-1.0,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0.000008,0.001314,0.000836,0.004810,0.003912,0.003198,0.001933,0.000000,0.047789,0.009482,0.002331,0.037110,0.009766,0.009766,0.002802,0.002802,0.066542,0.000044,0.074333,-1.0,0.002099,0.000000,-1.0,0.0,0.000374,0.000000,0.000000,0.000000,0.004044,-1.0,0.0,-1.0,0.000000,0.000000,0.0,0.0,0.002918,0.000016,0.000000
8,288,-1.000000,-1.0,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0.000072,0.002238,0.006232,0.004318,0.005901,0.004579,0.015637,0.024818,0.043750,0.011949,0.001415,0.047256,0.020659,0.020659,0.001585,0.001585,0.063264,0.000058,0.115384,-1.0,0.003801,0.000000,-1.0,0.0,0.000335,0.000000,0.000000,

In [3]:
# Initialize dataframe
column_names = ['ID', 'IDs', 'SFR', 'sSFR', 'stellar_mass', 'new_stellar_mass']
df = pd.DataFrame(columns = column_names)
# df.to_csv('sSFR_data100.csv', index=False)

In [3]:
# Load routine for getting tracks
def sSFR_track(ID=0, simulation=0, status=0, tangos_halo=0, halo_id=0, snap_num=0, resolution=1000):
    
#     from tangos_halo_module.halos import tangos_to_pynbody_halo, get_main_progenitor_branch

    if ID != 0:
        print('Started ', ID)
        simulation, status, halo_id, snap_num = ID_to_sim_halo_snap(ID=ID)
        print(simulation, status, halo_id, snap_num)
        tangos_halo = ID_to_tangos_halo(ID=ID, resolution=resolution)
    
    # Satellite MPB ids
    MPB, MPB_ids = get_main_progenitor_branch(tangos_halo=tangos_halo, simulation=simulation, resolution=resolution)
    print(MPB_ids)
    print(MPB)
    # Store and recall
    # Host MPB and ids
    host = get_host(simulation=simulation, resolution=resolution)
    host_MPB, host_ids = get_main_progenitor_branch(tangos_halo=host, simulation=simulation, 
                                                    resolution=resolution)

    # Only consider halo while separate from host
    MPB_ids[MPB_ids==host_ids] = 0
    print(MPB_ids)

    for i in range(len(MPB_ids)):
        if MPB_ids[i]!=0:
            if len(str(MPB[i]))>6:
                MPB_ids[i]=get_halo_snap_num(tangos_halo=MPB[i])[2]
            else: 
                MPB_ids[i]=0
    print('MPB new IDs: ', MPB_ids)
    IDs_track = np.zeros(0)
    SFR_track = np.zeros(0)
    sSFR_track = np.zeros(0)
    mass_track = np.zeros(0)
    new_mass_track = np.zeros(0)

    for index, idx in enumerate(MPB_ids):
        print(idx)
        if idx==0:
            IDs_track = np.concatenate((IDs_track, [-1]), axis=None)
            SFR_track = np.concatenate((SFR_track, [-1]), axis=None)
            sSFR_track = np.concatenate((sSFR_track, [-1]), axis=None)
            mass_track = np.concatenate((mass_track, [-1]), axis=None)
            new_mass_track = np.concatenate((new_mass_track, [-1]), axis=None)
        elif len(str(MPB[index]))<6:
            IDs_track = np.concatenate((IDs_track, [idx]), axis=None)
            SFR_track = np.concatenate((SFR_track, [-1]), axis=None)
            sSFR_track = np.concatenate((sSFR_track, [-1]), axis=None)
            mass_track = np.concatenate((mass_track, [-1]), axis=None)
            new_mass_track = np.concatenate((new_mass_track, [-1]), axis=None)
        else:
            tang_halo = ID_to_tangos_halo(ID=idx, resolution=resolution)
            print(tang_halo)
            pyn_halo = tangos_to_pynbody_halo(tangos_halo=tang_halo, simulation=simulation, resolution=resolution)
            mass = np.asarray(pyn_halo.s['mass'].in_units('Msol'))
            age = pyn_halo.properties['time'].in_units('Gyr') - pyn_halo.s['tform'].in_units('Gyr') #SimArray in Gyr
            new_mass = np.sum(mass[age <= 0.1]) # Formed within last 100 Myr
            total_mass = np.sum(mass)
            SFR = new_mass/1e8
            sSFR = SFR/total_mass
            IDs_track = np.concatenate((IDs_track, [idx]), axis=None)
            SFR_track = np.concatenate((SFR_track, [SFR]), axis=None)
            sSFR_track = np.concatenate((sSFR_track, [sSFR]), axis=None)
            mass_track = np.concatenate((mass_track, [total_mass]), axis=None)
            new_mass_track = np.concatenate((new_mass_track, [new_mass]), axis=None)
    print(SFR_track, sSFR_track, mass_track, new_mass_track)
    dat = {'ID': ID,
           'IDs': [IDs_track],
           'SFR': [SFR_track], 
           'sSFR': [sSFR_track], 
           'stellar_mass': [mass_track], 
           'new_stellar_mass': [new_mass_track],}
    df = pd.DataFrame(data=dat)
    return df
    
df = pd.read_csv('sSFR_data100.csv')
completed_ids = np.asarray(df['ID'])
len(completed_ids)

0

In [4]:
# Timing the slowest part of my code
# Test ID
# Well this works, so why does my loop keep breaking here??? (check pipe2.out for what I mean)

%timeit tang_halo = ID_to_tangos_halo(ID='1482784661', resolution=100)
tang_halo = ID_to_tangos_halo(ID='1482784661', resolution=100)
print(tang_halo)

%timeit pyn_halo = tangos_to_pynbody_halo(tangos_halo=tang_halo, simulation='h148', resolution=100)
pyn_halo = tangos_to_pynbody_halo(tangos_halo=tang_halo, simulation='h148', resolution=100)
print(pyn_halo)

%timeit mass = np.asarray(pyn_halo.s['mass'].in_units('Msol')) #This takes incredibly long
mass = np.asarray(pyn_halo.s['mass'].in_units('Msol')) #This takes incredibly long
print(mass)

%timeit age = pyn_halo.properties['time'].in_units('Gyr') - pyn_halo.s['tform'].in_units('Gyr') #SimArray in Gyr #This also takes forever
age = pyn_halo.properties['time'].in_units('Gyr') - pyn_halo.s['tform'].in_units('Gyr') #SimArray in Gyr #This also takes forever
print(age)

%timeit new_mass = np.sum(mass[age <= 0.1]) # Formed within last 100 Myr
new_mass = np.sum(mass[age <= 0.1]) # Formed within last 100 Myr
print(new_mass)

%timeit total_mass = np.sum(mass)
total_mass = np.sum(mass)
print(total_mass)

%timeit SFR = new_mass/1e8
SFR = new_mass/1e8
print(SFR)
sSFR = SFR/total_mass
print(sSFR)

# h2data = h.load_copy(2) @ https://pynbody.github.io/pynbody/tutorials/halos.html
# Reproduce quirm plots only for h148 and h329

100 loops, best of 3: 14.4 ms per loop
<Halo 'snapshots/h148.cosmo50PLK.6144g3HbwK1BH.002784/halo_661' | NDM=4191 Nstar=4 Ngas=0>
1 loop, best of 3: 2.52 s per loop
<SimSnap "/home/dp1144/tangos_simulations/h148/snapshots/h148.cosmo50PLK.6144g3HbwK1BH.002784:halo_661" len=4195>
The slowest run took 80.99 times longer than the fastest. This could mean that an intermediate result is being cached.
1 loop, best of 3: 2.08 s per loop
[    614.0498492      614.04995714  407982.27862832     614.04995714]
1 loop, best of 3: 1.53 s per loop
[ 9.16143659  9.15554392  9.61123794  9.15386031]
10000 loops, best of 3: 51.1 µs per loop
0.0
100000 loops, best of 3: 5.98 µs per loop
409824.428392
The slowest run took 18.00 times longer than the fastest. This could mean that an intermediate result is being cached.
10000000 loops, best of 3: 93.5 ns per loop
0.0
0.0


In [ ]:
count=0
done=0
while done<124:
    try:
        print('Watch me go for the ', count, 'th time...')
        for idx in ids:
            if idx in completed_ids:
                print('Already Exists ', idx)
                done=len(completed_ids)
                print('Completed Count --> ', done)
                pass
            else:
                print('Started ', idx)
                sat_df = sSFR_track(ID=idx, resolution=100)
                df = pd.concat([df, sat_df])
                df.to_csv('sSFR_data100.csv', index=False)
                completed_ids = np.concatenate((completed_ids, [idx]), axis=None)
                print('Finished ', idx)
                done=len(completed_ids)
                print('Completed Count --> ', done)
    except: 
        count+=1
        continue